# DORAnet → enzyme hypotheses → DNA design

This notebook converts DORAnet-generated reaction networks into concise downstream design tables: parsed reactions, enzyme hypotheses, enzyme-candidate templates, and non-operational DNA design plans for expert review.

In [1]:
from pathlib import Path
import glob
import json
import os
import re
import textwrap
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem
import time
import requests
from io import StringIO

/users/sghosh6/.conda/envs/doranet_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
from DORA_XGB import DORA_XGB
by_desc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_descending_MW')
by_asc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_ascending_MW')
add_concat_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_concat')
add_subtract_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_subtract')

/users/sghosh6/.conda/envs/doranet_env/lib/python3.10/site-packages/xgboost/compat.py:105: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [73]:
from Bio.Data import CodonTable

from dnachisel import (
    DnaOptimizationProblem,
    EnforceTranslation,
    EnforceGCContent,
    AvoidPattern,
    MaximizeCAI,
)

In [2]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
resultsDir = os.path.join(dataDir, 'Results/combinedEbolaVirus_bestMACAW_allDB_generative/')
DORANETmoleculesDataDir = os.path.join(resultsDir, 'DORAnet/InoAdeGuoXan/PATHWAY_reconstruction/')
DNADesignResultsDir = os.path.join(resultsDir, 'DNA_design/')
os.makedirs(DNADesignResultsDir, exist_ok=True)

print("Reading DORAnet results from:", DORANETmoleculesDataDir)
print("DNA design results will be saved at:", DNADesignResultsDir)

Reading DORAnet results from: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/PATHWAY_reconstruction/
DNA design results will be saved at: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/


In [3]:
# Subdirectories inside DNA_design
reactionResultsDir = os.path.join(DNADesignResultsDir, "01_reactions/")
moleculeResultsDir = os.path.join(DNADesignResultsDir, "02_molecules/")
enzymeMappingResultsDir = os.path.join(DNADesignResultsDir, "03_enzyme_mapping/")
routeResultsDir    = os.path.join(DNADesignResultsDir, "04_routes/")
sequenceResultsDir = os.path.join(DNADesignResultsDir, "05_uniprot_sequences/")
dnaResultsDir      = os.path.join(DNADesignResultsDir, "06_optimized_dna/")
handoffResultsDir  = os.path.join(DNADesignResultsDir, "07_webtool_handoff/")

# Create all directories
for dirPath in [
    DNADesignResultsDir,
    reactionResultsDir,
    moleculeResultsDir,
    DNADesignResultsDir,
    routeResultsDir,
    sequenceResultsDir,
    dnaResultsDir,
    handoffResultsDir,
]:
    os.makedirs(dirPath, exist_ok=True)

print("reactionResultsDir  :", reactionResultsDir)

reactionResultsDir  : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/01_reactions/


## 1. Discover DORAnet JSON files

In [4]:
DORANETmoleculesDataDir = Path(DORANETmoleculesDataDir)  

def extractFirstInt(text):
    match = re.search(r"\d+", text)
    return int(match.group(0)) if match else -1

# Find all JSON files anywhere under root
allJsonPaths = sorted(DORANETmoleculesDataDir.rglob("*.json"), key=lambda p: str(p).lower())

fileInfoList = []
for jsonPath in allJsonPaths:
    parentDir = jsonPath.parent
    fileInfoList.append({
        "dirName": parentDir.name,
        "dirPath": str(parentDir),
        "jsonName": jsonPath.name,
        "jsonPath": str(jsonPath),
        "folderNum": extractFirstInt(parentDir.name),  # optional helper field
    })

print(f"Root directory        : {DORANETmoleculesDataDir}")
print(f"JSON files found      : {len(fileInfoList)}")
print(f"Unique folders w/JSON : {len({x['dirPath'] for x in fileInfoList})}")

Root directory        : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/PATHWAY_reconstruction
JSON files found      : 20
Unique folders w/JSON : 20


## 2. Parse DORAnet reaction strings

In [5]:
def splitMoleculeString(moleculeString):
    return [mol for mol in str(moleculeString).split(".") if mol]

def parseDoranetReaction(rxnString, fileInfo):
    parts = str(rxnString).split(">")
    if len(parts) != 4:
        raise ValueError(f"Expected 4 fields separated by '>'; found {len(parts)}")

    reactants, ruleName, metaBlock, products = parts
    metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
    thermo, reactantStoich, productStoich, reactionType = metaParts

    return {
        "reactants": reactants,
        "products": products,
        "reactionString": f"{reactants} >> {products}",
        "ruleName": ruleName,
        "thermo": thermo,
        "reactantStoich": reactantStoich,
        "productStoich": productStoich,
        "reactionType": reactionType,
        "numReactantMolecules": len(splitMoleculeString(reactants)),
        "numProductMolecules": len(splitMoleculeString(products)),
        "sourceFolderNum": fileInfo.get("folderNum", -1),
        "sourceDirectory": fileInfo.get("dirName", ""),
        "sourceDirectoryPath": fileInfo.get("dirPath", ""),
        "sourceJsonName": fileInfo.get("jsonName", ""),
        "sourceJsonPath": fileInfo.get("jsonPath", ""),
    }

reactionRecords = []
jsonReadErrors = []
uniqueReactantMolecules = set()
uniqueProductMolecules = set()

for fileInfo in tqdm(fileInfoList, desc="Reading DORAnet JSON"):
    try:
        with open(fileInfo["jsonPath"], "r", encoding="utf-8") as f:
            reactionList = json.load(f)

        # handle both list and dict payloads
        if isinstance(reactionList, dict):
            # try common key names; otherwise fail clearly
            for k in ["reactions", "reactionList", "data"]:
                if k in reactionList and isinstance(reactionList[k], list):
                    reactionList = reactionList[k]
                    break
            else:
                raise ValueError("JSON is dict but no reaction list key found")

        if not isinstance(reactionList, list):
            raise ValueError(f"Expected list of reaction strings, got {type(reactionList)}")

        for rxnString in reactionList:
            record = parseDoranetReaction(rxnString, fileInfo)
            reactionRecords.append(record)
            uniqueReactantMolecules.update(splitMoleculeString(record["reactants"]))
            uniqueProductMolecules.update(splitMoleculeString(record["products"]))

    except Exception as exc:
        jsonReadErrors.append({**fileInfo, "error": str(exc)})

if not reactionRecords:
    raise RuntimeError("No reactions loaded. Check JSON discovery and reaction format.")

reactionDF = pd.DataFrame(reactionRecords).reset_index(drop=True)

# safer path handling
outputPath = Path(DNADesignResultsDir) / "reactionDF.csv"
reactionDF.to_csv(outputPath, index=False)

print(f"Reactions loaded     : {len(reactionDF):,}")
print(f"Failed JSON files    : {len(jsonReadErrors):,}")
print(f"Unique reactant mols : {len(uniqueReactantMolecules):,}")
print(f"Unique product mols  : {len(uniqueProductMolecules):,}")
print(f"Saved: {outputPath}")

reactionDF.head()

Reading DORAnet JSON: 100%|████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 24.21it/s]


Reactions loaded     : 176,400
Failed JSON files    : 0
Unique reactant mols : 283
Unique product mols  : 4,730
Saved: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/reactionDF.csv


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,sourceFolderNum,sourceDirectory,sourceDirectoryPath,sourceJsonName,sourceJsonPath
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...


### Keep only enzymatic DORAnet reactions

In [6]:
enzymaticReactionDF = reactionDF[
    reactionDF["reactionType"].astype(str).str.lower().str.contains(
        "enzyme|enzymatic|bio|biological",
        na=False
    )
].copy()

print(f"Total reactions: {len(reactionDF):,}")
print(f"Likely enzymatic reactions: {len(enzymaticReactionDF):,}")

enzymaticReactionDF.head()

Total reactions: 176,400
Likely enzymatic reactions: 176,400


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,sourceFolderNum,sourceDirectory,sourceDirectoryPath,sourceJsonName,sourceJsonPath
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...


### Use `DORA-XGB` to get feasibility score

In [40]:
reactionDF_DORAXGB = enzymaticReactionDF.copy()
#reactionDF_DORAXGB = enzymaticReactionDF.head(50).copy()

# Clean reaction string for model input
reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

def getFeasibilityScoresAndLabels(rxnStr):
    return pd.Series({
        "feasibilityScore_rule1": by_desc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule1": by_desc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule2": by_asc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule2": by_asc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule3": add_concat_model.predict_proba(rxnStr),
        "feasibilityLabel_rule3": add_concat_model.predict_label(rxnStr),

        "feasibilityScore_rule4": add_subtract_model.predict_proba(rxnStr),
        "feasibilityLabel_rule4": add_subtract_model.predict_label(rxnStr),
    })


reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

reactionDF_DORAXGB[
    [
        "feasibilityScore_rule1",
        "feasibilityLabel_rule1",
        "feasibilityScore_rule2",
        "feasibilityLabel_rule2",
        "feasibilityScore_rule3",
        "feasibilityLabel_rule3",
        "feasibilityScore_rule4",
        "feasibilityLabel_rule4",
    ]
] = reactionDF_DORAXGB["rxn_str"].apply(getFeasibilityScoresAndLabels)

reactionDF_DORAXGB

,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sourceJsonPath,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,0.000647,0.0,0.884856,1.0,0.729129,1.0,0.664541,1.0
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,0.000875,0.0,0.020655,0.0,0.001980,0.0,0.000030,0.0
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,0.477059,0.0,0.012712,0.0,0.327328,0.0,0.049994,0.0
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,0.191322,0.0,0.002167,0.0,0.000062,0.0,0.784046,1.0
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,0.506250,0.0,0.413190,0.0,0.227389,0.0,0.278599,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176395,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,COC(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H]...,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,rule0003_177,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,0.128867,0.0,0.179263,0.0,0.056613,0.0,0.003117,0.0
176396,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,CO[C@@H]1[C@H](OS(=O)(=O)O)[C@@H](C=O)O[C@H]1n...,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,rule0048_5,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,0.018481,0.0,0.958909,1.0,0.637625,1.0,0.646907,0.0
176397,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,C[C@]12O[C@@]1(CO)O[C@@H](n1cnc3c(=O)[nH]cnc31...,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,0.000203,0.0,0.001115,0.0,0.050057,0.0,0.026536,0.0
176398,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)O[C@H]1[C@H]2O[C@]2(n2cnc3c(=O)[nH]cnc32...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,rule0062_19,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,0.000353,0.0,0.291174,0.0,0.023250,0.0,0.017795,0.0


### Keep only `high feasible` reactions

In [43]:
reactionDF_DORAXGB_highFeasibility = (
    reactionDF_DORAXGB[
        (reactionDF_DORAXGB["feasibilityLabel_rule1"] == 1)
        & (reactionDF_DORAXGB["feasibilityScore_rule1"].notna())
    ]
    .sort_values(by="feasibilityScore_rule1", ascending=False)
    .reset_index(drop=True)
)


uniqueReactantStrings_high = set(reactionDF_DORAXGB_highFeasibility["reactants"])
uniqueProductStrings_high = set(reactionDF_DORAXGB_highFeasibility["products"])

uniqueReactantMolecules_high = {
    mol
    for reactants in reactionDF_DORAXGB_highFeasibility["reactants"]
    for mol in str(reactants).split(".")
}

uniqueProductMolecules_high = {
    mol
    for products in reactionDF_DORAXGB_highFeasibility["products"]
    for mol in str(products).split(".")
}

print(f"Number of high-feasibility reactions: {len(reactionDF_DORAXGB_highFeasibility)}")
print(f"Number of unique reactant strings: {len(uniqueReactantStrings_high)}")
print(f"Number of unique product strings: {len(uniqueProductStrings_high)}")
print(f"Number of unique individual reactant molecules: {len(uniqueReactantMolecules_high)}")
print(f"Number of unique individual product molecules: {len(uniqueProductMolecules_high)}")
reactionDF_DORAXGB_highFeasibility.to_csv(os.path.join(DNADesignResultsDir, "DORAXGB_highFeasibility_reactions.csv"),index=False)
reactionDF_DORAXGB_highFeasibility

Number of high-feasibility reactions: 17400
Number of unique reactant strings: 564
Number of unique product strings: 705
Number of unique individual reactant molecules: 241
Number of unique individual product molecules: 640


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sourceJsonPath,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
1,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
2,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
3,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
4,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17395,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17396,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17397,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17398,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0


In [45]:
reactionDF_DORAXGB_highFeasibility = pd.read_csv(os.path.join(DNADesignResultsDir, "DORAXGB_highFeasibility_reactions.csv"))
reactionDF_DORAXGB_highFeasibility

,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sourceJsonPath,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
1,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
2,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
3,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
4,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17395,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17396,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17397,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17398,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0


### Summarize unique DORAnet rules

Many reactions may use the same rule. We do not want to annotate millions of reactions one by one. First annotate the rules.

In [47]:
enzymaticReactionDF = reactionDF_DORAXGB_highFeasibility.copy()
ruleSummaryDF = (
    enzymaticReactionDF
    .groupby(["ruleName", "reactionType"], dropna=False)
    .agg(
        numReactions=("reactionString", "count"),
        numStarterSources=("sourceFolderNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("reactants", "first"),
        exampleProducts=("products", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

print(f"unique DORAnet rules    : {enzymaticReactionDF['ruleName'].nunique():,}")
ruleSummaryDF

unique DORAnet rules    : 68


,ruleName,reactionType,numReactions,numStarterSources,exampleReaction,exampleReactants,exampleProducts
22,rule0011_51,Enzymatic,3460,20,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C)nc32...
21,rule0011_50,Enzymatic,2360,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...
42,rule0043_12,Enzymatic,2020,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,CC(C(=O)O)C(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]...
52,rule0121_1,Enzymatic,1520,20,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...
67,rule0491_2,Enzymatic,1380,20,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H]...
...,...,...,...,...,...,...,...
46,rule0062_17,Enzymatic,20,20,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)OC(C)=O.CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]...
15,rule0009_40,Enzymatic,20,20,CC(=O)O.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,CC(=O)O.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,CC(=O)O[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1...
16,rule0009_41,Enzymatic,20,20,CO.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O[...,CO.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O[...,CO[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O.O=c...
65,rule0394_7,Enzymatic,20,20,N#CC(O)(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,N#CC(O)(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,C#N.O=C(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...


### Add required placeholder columns

In [48]:
enzymeAnnotationDF = ruleSummaryDF.copy()

enzymeAnnotationDF["suggestedEnzymeClass"] = ""
enzymeAnnotationDF["ecNumber"] = ""
enzymeAnnotationDF["enzymeName"] = ""
enzymeAnnotationDF["uniprotAccession"] = ""
enzymeAnnotationDF["sourceDatabase"] = ""
enzymeAnnotationDF["reactionSimilarity"] = ""
enzymeAnnotationDF["enzymeConfidence"] = ""
enzymeAnnotationDF["notes"] = ""
enzymeAnnotationDF.head()

,ruleName,reactionType,numReactions,numStarterSources,exampleReaction,exampleReactants,exampleProducts,suggestedEnzymeClass,ecNumber,enzymeName,uniprotAccession,sourceDatabase,reactionSimilarity,enzymeConfidence,notes
22,rule0011_51,Enzymatic,3460,20,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C)nc32...,,,,,,,,
21,rule0011_50,Enzymatic,2360,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,,,,,,,,
42,rule0043_12,Enzymatic,2020,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,CC(C(=O)O)C(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]...,,,,,,,,
52,rule0121_1,Enzymatic,1520,20,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,,,,,,,,
67,rule0491_2,Enzymatic,1380,20,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H]...,,,,,,,,


## 3. Read DORAnet reaction ruleset file

In [49]:
doranetRulesetDF = pd.read_csv(dataDir + "/DORAnet/JN3604IMT_rules.tsv", sep="\t")

print(doranetRulesetDF.shape)
print(doranetRulesetDF.columns.tolist())
doranetRulesetDF.head()

(3604, 5)
['Name', 'Reactants', 'SMARTS', 'Products', 'Comments']


,Name,Reactants,SMARTS,Products,Comments
0,rule0001_01,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P97259;P97402;Q08834;Q09328;Q16408;Q3V5L5;Q765...
1,rule0001_02,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A0K8D0;A0KHH6;A0KZ10;A0L8R9;A0Q7X9;A1A7M6;A1K6...
2,rule0001_03,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P0DMP6;P0DMP7;P16442;Q15951;Q1RLK6;Q3USF0;Q5HZ...
3,rule0001_04,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A4IID1;O19071;O77836;P23336;P26572;P27115;P278...
4,rule0001_05,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,NaN


### Find the number of common reaction rules between `DORAnet` diversification and ruleset table from actual package

In [50]:
reactionRuleSet = set(reactionDF_DORAXGB_highFeasibility["ruleName"].dropna().astype(str).str.strip())
tsvRuleSet = set(doranetRulesetDF["Name"].dropna().astype(str).str.strip())

matchedRuleSet = reactionRuleSet.intersection(tsvRuleSet)
missingRuleSet = reactionRuleSet.difference(tsvRuleSet)

print(f"Rules present in DORAnet diversification run: {len(reactionRuleSet):,}")
print(f"Rules matched from DORAnet package: {len(matchedRuleSet):,}")
print(f"Rules missing from DORAnet package: {len(missingRuleSet):,}")

print("\nMatched rules:")
print(sorted(list(matchedRuleSet))[:20])

print("\nMissing rules:")
print(sorted(list(missingRuleSet))[:20])

Rules present in DORAnet diversification run: 68
Rules matched from DORAnet package: 68
Rules missing from DORAnet package: 0

Matched rules:
['rule0001_88', 'rule0003_170', 'rule0003_175', 'rule0003_176', 'rule0003_177', 'rule0004_13', 'rule0007_161', 'rule0007_174', 'rule0007_195', 'rule0007_198', 'rule0007_199', 'rule0007_200', 'rule0007_201', 'rule0007_202', 'rule0007_203', 'rule0009_40', 'rule0009_41', 'rule0010_64', 'rule0010_65', 'rule0011_48']

Missing rules:
[]


### Extract `UniProt IDs` from `DORAnet` reaction rules

In [51]:
def splitUniProtIds(idString):
    if pd.isna(idString):
        return []
    return [x.strip() for x in str(idString).split(";") if x.strip()]


ruleInfoDF = doranetRulesetDF.copy()

ruleInfoDF = ruleInfoDF.rename(columns={
    "Name": "ruleName",
    "Reactants": "ruleReactants",
    "SMARTS": "ruleSMARTS",
    "Products": "ruleProducts",
    "Comments": "candidateUniProtRaw",
})

ruleInfoDF["ruleName"] = ruleInfoDF["ruleName"].astype(str).str.strip()
ruleInfoDF["candidateUniProtList"] = ruleInfoDF["candidateUniProtRaw"].apply(splitUniProtIds)
ruleInfoDF["numCandidateUniProt"] = ruleInfoDF["candidateUniProtList"].apply(len)

ruleInfoDF = ruleInfoDF[
    [
        "ruleName",
        "ruleReactants",
        "ruleProducts",
        "ruleSMARTS",
        "candidateUniProtRaw",
        "numCandidateUniProt",
    ]
].copy()



print(f"ruleInfoDF rows: {len(ruleInfoDF):,}")
ruleInfoDF

ruleInfoDF rows: 3,604


,ruleName,ruleReactants,ruleProducts,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt
0,rule0001_01,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,P97259;P97402;Q08834;Q09328;Q16408;Q3V5L5;Q765...,10
1,rule0001_02,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,A0K8D0;A0KHH6;A0KZ10;A0L8R9;A0Q7X9;A1A7M6;A1K6...,263
2,rule0001_03,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,P0DMP6;P0DMP7;P16442;Q15951;Q1RLK6;Q3USF0;Q5HZ...,29
3,rule0001_04,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,A4IID1;O19071;O77836;P23336;P26572;P27115;P278...,42
4,rule0001_05,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,NaN,0
...,...,...,...,...,...,...
3599,rule1152_1,Any;Any,Any;Any,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,Q8ZUN8,1
3600,rule1152_2,Any;Any,Any;Any,[#6;!$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[...,B9TTF1,1
3601,rule1164_1,Any;WATER,Any;H2O2,[#6;$([#6&!R]-&!@[#6&R]1:&@[#6&R]:&@[#6&R]:&@[...,Q972I2,1
3602,rule1165_1,Any;H2O2,Any;WATER,[#6;$([#6&!R]-[#6&R]1:&@[#6&R]:&@[#6&R]:&@[#6&...,Q972I2,1


## 4. Merge rule information into `reactionDF`

This connects DORAnet reactions with the rule SMARTS and UniProt candidate list.

In [52]:
reactionWithRuleDF = reactionDF_DORAXGB_highFeasibility.merge(ruleInfoDF,on="ruleName",how="left")

reactionWithRuleDF["hasRuleLookup"] = reactionWithRuleDF["ruleSMARTS"].notna()

print(reactionWithRuleDF["hasRuleLookup"].value_counts(dropna=False))
reactionWithRuleDF = reactionWithRuleDF.drop(columns=['sourceDirectory', 'sourceDirectoryPath', 'sourceJsonName', 'sourceJsonPath'])
reactionWithRuleDF

hasRuleLookup
True    17400
Name: count, dtype: int64


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4,ruleReactants,ruleProducts,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt,hasRuleLookup
0,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.316587,0.0,0.438167,0.0,METHYL_DONOR_CoF;Any,METHYL_ACCEPTOR_CoF;Any,[#6:1]-[#16+:2].[#8;$([#8&!R]-[#6&!R]);!$([#8&...,A6XNE5,1,True
1,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.316587,0.0,0.438167,0.0,METHYL_DONOR_CoF;Any,METHYL_ACCEPTOR_CoF;Any,[#6:1]-[#16+:2].[#8;$([#8&!R]-[#6&!R]);!$([#8&...,A6XNE5,1,True
2,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.316587,0.0,0.438167,0.0,METHYL_DONOR_CoF;Any,METHYL_ACCEPTOR_CoF;Any,[#6:1]-[#16+:2].[#8;$([#8&!R]-[#6&!R]);!$([#8&...,A6XNE5,1,True
3,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.316587,0.0,0.438167,0.0,METHYL_DONOR_CoF;Any,METHYL_ACCEPTOR_CoF;Any,[#6:1]-[#16+:2].[#8;$([#8&!R]-[#6&!R]);!$([#8&...,A6XNE5,1,True
4,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.316587,0.0,0.438167,0.0,METHYL_DONOR_CoF;Any,METHYL_ACCEPTOR_CoF;Any,[#6:1]-[#16+:2].[#8;$([#8&!R]-[#6&!R]);!$([#8&...,A6XNE5,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17395,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.063748,0.0,0.192906,0.0,Any;METHYL_DONOR_CoF,METHYL_ACCEPTOR_CoF;Any,[#6;$([#6&!R]-[#6&!R]):1].[#6:2]-[#16+:3]>>[#1...,A3KI18;A4FG18;C4R7Z3;D3KYU3;D5FKJ3;E9NH27;E9NH...,17,True
17396,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.063748,0.0,0.192906,0.0,Any;METHYL_DONOR_CoF,METHYL_ACCEPTOR_CoF;Any,[#6;$([#6&!R]-[#6&!R]):1].[#6:2]-[#16+:3]>>[#1...,A3KI18;A4FG18;C4R7Z3;D3KYU3;D5FKJ3;E9NH27;E9NH...,17,True
17397,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.063748,0.0,0.192906,0.0,Any;METHYL_DONOR_CoF,METHYL_ACCEPTOR_CoF;Any,[#6;$([#6&!R]-[#6&!R]):1].[#6:2]-[#16+:3]>>[#1...,A3KI18;A4FG18;C4R7Z3;D3KYU3;D5FKJ3;E9NH27;E9NH...,17,True
17398,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.063748,0.0,0.192906,0.0,Any;METHYL_DONOR_CoF,METHYL_ACCEPTOR_CoF;Any,[#6;$([#6&!R]-[#6&!R]):1].[#6:2]-[#16+:3]>>[#1...,A3KI18;A4FG18;C4R7Z3;D3KYU3;D5FKJ3;E9NH27;E9NH...,17,True


### Summarize used rules to understand which rules are most frequent in DORAnet network and how many candidate UniProt proteins each rule has

In [54]:
usedRuleSummaryDF = (
    reactionWithRuleDF[
        reactionWithRuleDF["hasRuleLookup"]
    ]
    .groupby(
        [
            "ruleName",
            "reactionType",
            "ruleReactants",
            "ruleProducts",
            "ruleSMARTS",
            "candidateUniProtRaw",
            "numCandidateUniProt",
        ],
        dropna=False
    )
    .agg(
        numReactions=("reactionString", "count"),
        numStarterSources=("sourceFolderNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("reactants", "first"),
        exampleProducts=("products", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)


print(f"Used matched rules: {len(usedRuleSummaryDF):,}")
usedRuleSummaryDF[
    [
        "ruleName",
        "reactionType",
        "numReactions",
        "numCandidateUniProt",
        "exampleReaction",
    ]
]

Used matched rules: 68


,ruleName,reactionType,numReactions,numCandidateUniProt,exampleReaction
22,rule0011_51,Enzymatic,3460,1,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...
21,rule0011_50,Enzymatic,2360,1,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...
42,rule0043_12,Enzymatic,2020,17,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...
52,rule0121_1,Enzymatic,1520,932,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...
67,rule0491_2,Enzymatic,1380,0,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...
...,...,...,...,...,...
46,rule0062_17,Enzymatic,20,211,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...
15,rule0009_40,Enzymatic,20,8,CC(=O)O.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)...
16,rule0009_41,Enzymatic,20,2,CO.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O[...
65,rule0394_7,Enzymatic,20,4,N#CC(O)(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...


## Build one row per rule–UniProt candidate

This converts: `ruleName → "P12345;Q9XYZ1;...`

into: `ruleName | uniprotAccession`

In [55]:
usedRuleSet = set(
    reactionWithRuleDF.loc[
        reactionWithRuleDF["hasRuleLookup"],
        "ruleName"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

usedRuleInfoDF = ruleInfoDF[
    ruleInfoDF["ruleName"].isin(usedRuleSet)
].copy()

ruleCandidateRecords = []

for _, row in usedRuleInfoDF.iterrows():
    candidateList = splitUniProtIds(row["candidateUniProtRaw"])

    for rankIndex, uniprotAccession in enumerate(candidateList, start=1):
        ruleCandidateRecords.append({
            "ruleName": row["ruleName"],
            "candidateRankInRuleFile": rankIndex,
            "uniprotAccession": uniprotAccession,
            "ruleReactants": row["ruleReactants"],
            "ruleProducts": row["ruleProducts"],
            "ruleSMARTS": row["ruleSMARTS"],
        })

usedRuleUniProtCandidateDF = pd.DataFrame(ruleCandidateRecords)

usedRuleUniProtCandidateDF.to_csv(
    os.path.join(DNADesignResultsDir, "usedRuleUniProtCandidateDF.csv"),
    index=False
)

print(f"Used rule-UniProt candidate rows: {len(usedRuleUniProtCandidateDF):,}")
usedRuleUniProtCandidateDF

Used rule-UniProt candidate rows: 11,538


,ruleName,candidateRankInRuleFile,uniprotAccession,ruleReactants,ruleProducts,ruleSMARTS
0,rule0001_88,1,Q494Q2,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...
1,rule0001_88,2,Q84IF9,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...
2,rule0001_88,3,Q9FY51,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...
3,rule0003_170,1,A1CFL1,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...
4,rule0003_170,2,A1L4Y2,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...
...,...,...,...,...,...,...
11533,rule0467_5,4,P13650,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...
11534,rule0467_5,5,P15877,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...
11535,rule0467_5,6,P27175,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...
11536,rule0467_5,7,Q8ZUN8,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...


### Select subset of UniProt candidates for metadata retrieval

Some DORAnet rules can map to hundreds or thousands of UniProt candidates. For a first query only the top `maxCandidatesPerRule` candidates per rule need to be selcted.

In [56]:
maxCandidatesPerRule = 50

candidateSubsetDF = (
    usedRuleUniProtCandidateDF
    .sort_values(["ruleName", "candidateRankInRuleFile"])
    .groupby("ruleName")
    .head(maxCandidatesPerRule)
    .reset_index(drop=True)
)

print(f"Rules represented            : {candidateSubsetDF['ruleName'].nunique():,}")
print(f"Candidate rows to query      : {len(candidateSubsetDF):,}")
print(f"Unique UniProt IDs to query  : {candidateSubsetDF['uniprotAccession'].nunique():,}")
candidateSubsetDF

Rules represented            : 64
Candidate rows to query      : 1,956
Unique UniProt IDs to query  : 1,642


,ruleName,candidateRankInRuleFile,uniprotAccession,ruleReactants,ruleProducts,ruleSMARTS
0,rule0001_88,1,Q494Q2,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...
1,rule0001_88,2,Q84IF9,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...
2,rule0001_88,3,Q9FY51,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...
3,rule0003_170,1,A1CFL1,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...
4,rule0003_170,2,A1L4Y2,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...
...,...,...,...,...,...,...
1951,rule0467_5,4,P13650,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...
1952,rule0467_5,5,P15877,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...
1953,rule0467_5,6,P27175,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...
1954,rule0467_5,7,Q8ZUN8,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...


## 6. Query UniProt for EC numbers, protein names, organisms and sequences

```text
DORAnet rule ID → UniProt accession → EC number/protein sequence
```

In [57]:
# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def chunkList(inputList, chunkSize):
    for i in range(0, len(inputList), chunkSize):
        yield inputList[i:i + chunkSize]


def cleanUniProtAccession(x):
    """
    Clean UniProt accession IDs coming from DORAnet/JN3604IMT Comments column.
    Examples:
        UniProtKB:P12345 -> P12345
        P12345-2         -> P12345
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    # Remove common prefixes
    x = x.replace("UniProtKB:", "")
    x = x.replace("UniProt:", "")
    x = x.replace("uniprot:", "")

    # Keep only first token if accidental text is present
    x = x.split()[0].strip()

    # Use canonical accession rather than isoform-specific accession
    # Example: P12345-2 -> P12345
    x = x.split("-")[0].strip()

    if x == "":
        return None

    return x


# UniProt accession pattern: supports common 6-character and 10-character accessions
uniprotAccessionPattern = re.compile(
    r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z0-9]{3}[0-9]){1,2})$"
)


def isValidUniProtAccession(x):
    if pd.isna(x):
        return False
    return bool(uniprotAccessionPattern.match(str(x).strip()))


def fetchUniProtMetadataFixed(accessionList, chunkSize=20, sleepSeconds=0.5):
    """
    Query UniProtKB metadata for a list of UniProt accessions.

    Important:
    Use accession_id:{acc}, not accession:{acc}.
    """
    metadataDFList = []
    errorRecords = []

    fields = [
        "accession",
        "reviewed",
        "id",
        "protein_name",
        "gene_names",
        "organism_name",
        "organism_id",
        "ec",
        "length",
        "sequence",
    ]

    accessionChunkList = list(chunkList(accessionList, chunkSize))

    for accessionChunk in tqdm(accessionChunkList, desc="Querying UniProt"):
        query = " OR ".join([f"(accession_id:{acc})" for acc in accessionChunk])

        url = "https://rest.uniprot.org/uniprotkb/search"

        params = {
            "query": query,
            "format": "tsv",
            "fields": ",".join(fields),
            "size": chunkSize,
        }

        try:
            response = requests.get(url, params=params, timeout=120)

            if response.status_code != 200:
                errorRecords.append({
                    "chunkFirstAccession": accessionChunk[0],
                    "chunkSize": len(accessionChunk),
                    "statusCode": response.status_code,
                    "message": response.text[:1000],
                    "requestUrl": response.url,
                })
                continue

            chunkDF = pd.read_csv(StringIO(response.text), sep="\t")

            if len(chunkDF) == 0:
                errorRecords.append({
                    "chunkFirstAccession": accessionChunk[0],
                    "chunkSize": len(accessionChunk),
                    "statusCode": response.status_code,
                    "message": "Request succeeded but returned zero rows.",
                    "requestUrl": response.url,
                })
                continue

            metadataDFList.append(chunkDF)

        except Exception as exc:
            errorRecords.append({
                "chunkFirstAccession": accessionChunk[0],
                "chunkSize": len(accessionChunk),
                "statusCode": None,
                "message": str(exc),
                "requestUrl": None,
            })

        time.sleep(sleepSeconds)

    if metadataDFList:
        metadataDF = pd.concat(metadataDFList, ignore_index=True)
        metadataDF = metadataDF.drop_duplicates()
    else:
        metadataDF = pd.DataFrame()

    errorDF = pd.DataFrame(errorRecords)

    return metadataDF, errorDF


# ------------------------------------------------------------
# 2. Clean candidate UniProt IDs from candidateSubsetDF
# ------------------------------------------------------------

candidateSubsetDF = candidateSubsetDF.copy()

candidateSubsetDF["uniprotAccessionClean"] = (
    candidateSubsetDF["uniprotAccession"]
    .apply(cleanUniProtAccession)
)

candidateSubsetDF["isValidUniProtAccession"] = (
    candidateSubsetDF["uniprotAccessionClean"]
    .apply(isValidUniProtAccession)
)

print("Valid/invalid UniProt accession counts:")
print(candidateSubsetDF["isValidUniProtAccession"].value_counts(dropna=False))


# Save invalid UniProt accessions for inspection
invalidUniProtDF = candidateSubsetDF[
    ~candidateSubsetDF["isValidUniProtAccession"]
].copy()

invalidUniProtDF.to_csv(
    os.path.join(sequenceResultsDir, "invalidUniProtAccessionsDF.csv"),
    index=False
)

print(f"Invalid UniProt rows saved: {len(invalidUniProtDF):,}")


# ------------------------------------------------------------
# 3. Build clean unique UniProt accession list
# ------------------------------------------------------------

uniqueAccessionList = sorted(
    candidateSubsetDF.loc[
        candidateSubsetDF["isValidUniProtAccession"],
        "uniprotAccessionClean"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"Valid unique UniProt accessions to query: {len(uniqueAccessionList):,}")


# ------------------------------------------------------------
# 4. Test the UniProt query on one accession first
# ------------------------------------------------------------

if len(uniqueAccessionList) == 0:
    raise RuntimeError("No valid UniProt accessions found after cleaning.")

testAccessionList = uniqueAccessionList[:1]

testMetadataDF, testErrorDF = fetchUniProtMetadataFixed(
    accessionList=testAccessionList,
    chunkSize=1,
    sleepSeconds=0.2,
)

print("\nTest query result:")
print(f"Test metadata rows: {len(testMetadataDF):,}")
print(f"Test error rows   : {len(testErrorDF):,}")

if len(testErrorDF) > 0:
    print("\nTest error message:")
    print(testErrorDF[["statusCode", "message"]].head(1).to_string(index=False))
    raise RuntimeError("UniProt test query failed. Check the error message above.")

display(testMetadataDF)


# ------------------------------------------------------------
# 5. Run the full UniProt metadata query
# ------------------------------------------------------------

uniprotMetadataRawDF, uniprotQueryErrorDF = fetchUniProtMetadataFixed(
    accessionList=uniqueAccessionList,
    chunkSize=20,
    sleepSeconds=0.5,
)

uniprotMetadataRawDF.to_csv(
    os.path.join(sequenceResultsDir, "uniprotMetadataRawDF.csv"),
    index=False
)

if len(uniprotQueryErrorDF) > 0:
    uniprotQueryErrorDF.to_csv(
        os.path.join(sequenceResultsDir, "uniprotQueryErrorDF.csv"),
        index=False
    )

print("\nFull query result:")
print(f"Retrieved UniProt metadata rows: {len(uniprotMetadataRawDF):,}")
print(f"Query error chunks             : {len(uniprotQueryErrorDF):,}")

if len(uniprotQueryErrorDF) > 0:
    print("\nExample query errors:")
    display(uniprotQueryErrorDF.head())

display(uniprotMetadataRawDF.head())


# ------------------------------------------------------------
# 6. Standardize UniProt metadata column names
# ------------------------------------------------------------

uniprotMetadataDF = uniprotMetadataRawDF.copy()

columnRenameDict = {
    "Entry": "uniprotAccession",
    "Reviewed": "uniprotReviewed",
    "Entry Name": "entryName",
    "Protein names": "proteinName",
    "Gene Names": "geneNames",
    "Organism": "organism",
    "Organism (ID)": "organismTaxId",
    "EC number": "ecNumber",
    "Length": "proteinLengthAa",
    "Sequence": "proteinSequence",
}

uniprotMetadataDF = uniprotMetadataDF.rename(columns=columnRenameDict)

requiredUniProtCols = [
    "uniprotAccession",
    "uniprotReviewed",
    "entryName",
    "proteinName",
    "geneNames",
    "organism",
    "organismTaxId",
    "ecNumber",
    "proteinLengthAa",
    "proteinSequence",
]

for colName in requiredUniProtCols:
    if colName not in uniprotMetadataDF.columns:
        uniprotMetadataDF[colName] = np.nan

uniprotMetadataDF = uniprotMetadataDF[requiredUniProtCols].copy()

uniprotMetadataDF.to_csv(
    os.path.join(sequenceResultsDir, "uniprotMetadataDF.csv"),
    index=False
)

print("\nStandardized UniProt metadata:")
print(uniprotMetadataDF.shape)
display(uniprotMetadataDF.head())


# ------------------------------------------------------------
# 7. Merge UniProt metadata back to rule candidates
# ------------------------------------------------------------

candidateSubsetCleanDF = candidateSubsetDF[
    candidateSubsetDF["isValidUniProtAccession"]
].copy()

candidateSubsetCleanDF = candidateSubsetCleanDF.rename(columns={
    "uniprotAccession": "uniprotAccessionOriginal",
    "uniprotAccessionClean": "uniprotAccession",
})

ruleCandidateAnnotatedDF = candidateSubsetCleanDF.merge(
    uniprotMetadataDF,
    on="uniprotAccession",
    how="left"
)

ruleCandidateAnnotatedDF["hasEcNumber"] = (
    ruleCandidateAnnotatedDF["ecNumber"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

ruleCandidateAnnotatedDF["hasProteinSequence"] = (
    ruleCandidateAnnotatedDF["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

ruleCandidateAnnotatedDF.to_csv(
    os.path.join(DNADesignResultsDir, "ruleCandidateAnnotatedDF.csv"),
    index=False
)

print("\nAnnotated rule-candidate summary:")
print("EC number availability:")
print(ruleCandidateAnnotatedDF["hasEcNumber"].value_counts(dropna=False))

print("\nProtein sequence availability:")
print(ruleCandidateAnnotatedDF["hasProteinSequence"].value_counts(dropna=False))

display(ruleCandidateAnnotatedDF.head())


# ------------------------------------------------------------
# 8. Optional: summarize missing UniProt metadata
# ------------------------------------------------------------

missingMetadataDF = ruleCandidateAnnotatedDF[
    ruleCandidateAnnotatedDF["proteinName"].isna()
].copy()

missingMetadataDF.to_csv(
    os.path.join(sequenceResultsDir, "missingUniProtMetadataDF.csv"),
    index=False
)

print(f"\nRows with missing UniProt metadata: {len(missingMetadataDF):,}")

Valid/invalid UniProt accession counts:
isValidUniProtAccession
True     1944
False      12
Name: count, dtype: int64
Invalid UniProt rows saved: 12
Valid unique UniProt accessions to query: 1,635


Querying UniProt: 100%|██████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.01s/it]


Test query result:
Test metadata rows: 1
Test error rows   : 0


,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Organism (ID),EC number,Length,Sequence
0,A0A060TBM3,unreviewed,A0A060TBM3_BLAAD,alcohol dehydrogenase (EC 1.1.1.1),AADH1 GNLVRS02_ARAD1B16786g,Blastobotrys adeninivorans (Yeast) (Arxula ade...,409370,1.1.1.1,348,MSIPKTQKAVVFDKNGGPLTYKDIPVPEPADDQILINVKYSGVCHT...


Querying UniProt: 100%|████████████████████████████████████████████████████████████████████████████████████| 82/82 [01:43<00:00,  1.26s/it]

...


Standardized UniProt metadata:
(1635, 10)


,uniprotAccession,uniprotReviewed,entryName,proteinName,geneNames,organism,organismTaxId,ecNumber,proteinLengthAa,proteinSequence
0,A0A0D2YG05,reviewed,FUB5_FUSO4,Homoserine O-acetyltransferase FUB5 (EC 2.3.1....,FUB5 FOXG_15243,Fusarium oxysporum f. sp. lycopersici (strain ...,426428.0,2.3.1.31,362.0,MSWGKLSPKANNVMIICHALSGSADVSDWWGPLLGPGKAFDTDKFF...
1,A0A1D8PP43,unreviewed,A0A1D8PP43_CANAL,alcohol dehydrogenase (EC 1.1.1.1),ADH1 CAALFM_C505050WA orf19.11480,Candida albicans (strain SC5314 / ATCC MYA-287...,237561.0,1.1.1.1,350.0,MSEQIPKTQKAVVFDTNGGQLVYKDYPVPTPKPNELLIHVKYSGVC...
2,A0A1D3TPC3,unreviewed,A0A1D3TPC3_9FIRM,"2,3-bisphosphoglycerate-independent phosphogly...",gpmI SAMN05421730_1001389,Anaerobium acetethylicum,1619234.0,5.4.2.12,514.0,MSKKPTVLMILDGYGLNERTDGNAIAEAKTPVISRLMKEYPFVKGY...
3,A0A0J9X7D2,unreviewed,A0A0J9X7D2_GEOCN,Histidine biosynthesis trifunctional protein [...,BN980_GECA03s06082g DV451_001508,Geotrichum candidum (Oospora lactis) (Dipodasc...,1173061.0,1.1.1.23; 3.5.4.19; 3.6.1.31,866.0,MFPLVPLTTLSQIKAAEAVAAPTGRLLVVAEAANYKTDLPQYIKAN...
4,A0A060TBM3,unreviewed,A0A060TBM3_BLAAD,alcohol dehydrogenase (EC 1.1.1.1),AADH1 GNLVRS02_ARAD1B16786g,Blastobotrys adeninivorans (Yeast) (Arxula ade...,409370.0,1.1.1.1,348.0,MSIPKTQKAVVFDKNGGPLTYKDIPVPEPADDQILINVKYSGVCHT...



Annotated rule-candidate summary:
EC number availability:
hasEcNumber
True     1764
False     180
Name: count, dtype: int64

Protein sequence availability:
hasProteinSequence
True     1914
False      30
Name: count, dtype: int64


,ruleName,candidateRankInRuleFile,uniprotAccessionOriginal,ruleReactants,ruleProducts,ruleSMARTS,uniprotAccession,isValidUniProtAccession,uniprotReviewed,entryName,proteinName,geneNames,organism,organismTaxId,ecNumber,proteinLengthAa,proteinSequence,hasEcNumber,hasProteinSequence
0,rule0001_88,1,Q494Q2,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...,Q494Q2,True,reviewed,HPAT2_ARATH,Hydroxyproline O-arabinosyltransferase 2 (EC 2...,HPAT2 At2g25260 T22F11.15,Arabidopsis thaliana (Mouse-ear cress),3702.0,2.4.2.58,358.0,MGFRGKYFFPILMTLSLFLIIRYNYIVSDDPPLRQELPGRRSASSG...,True,True
1,rule0001_88,2,Q84IF9,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...,Q84IF9,True,unreviewed,Q84IF9_GEOSE,Cysteine synthase (EC 2.5.1.47),NaN,Geobacillus stearothermophilus (Bacillus stear...,1422.0,2.5.1.47,308.0,MARTVNSITELIGDTPAVKLNRIVDEDSADVYLKLEFMNPGSSVKD...,True,True
2,rule0001_88,3,Q9FY51,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...,Q9FY51,True,reviewed,HPAT3_ARATH,Hydroxyproline O-arabinosyltransferase 3 (EC 2...,HPAT3 At5g13500 T6I14.30,Arabidopsis thaliana (Mouse-ear cress),3702.0,2.4.2.58,358.0,MGKASGLLLFLLGFGFFVVTYNLLTLIVHNRSGVSNSDGSPLLDPV...,True,True
3,rule0003_170,1,A1CFL1,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...,A1CFL1,True,reviewed,PATD_ASPCL,Alcohol dehydrogenase patD (EC 1.1.1.-) (Patul...,patD ACLA_093590,Aspergillus clavatus (strain ATCC 1007 / CBS 5...,344612.0,1.1.1.-,388.0,MGSTLPTTYKRAFFEKQDATLTLEEVQLIEPQRGEILVKVEACGVC...,True,True
4,rule0003_170,2,A1L4Y2,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...,A1L4Y2,True,reviewed,ADHL3_ARATH,Alcohol dehydrogenase-like 3 (EC 1.1.1.1),At1g32780 F6N18.16,Arabidopsis thaliana (Mouse-ear cress),3702.0,1.1.1.1,394.0,MAETQGKVITCKAAVVWGPKVPLVIQEICVDPPQKMEVRVKILYSS...,True,True



Rows with missing UniProt metadata: 0


In [58]:
ruleCandidateAnnotatedDF

,ruleName,candidateRankInRuleFile,uniprotAccessionOriginal,ruleReactants,ruleProducts,ruleSMARTS,uniprotAccession,isValidUniProtAccession,uniprotReviewed,entryName,proteinName,geneNames,organism,organismTaxId,ecNumber,proteinLengthAa,proteinSequence,hasEcNumber,hasProteinSequence
0,rule0001_88,1,Q494Q2,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...,Q494Q2,True,reviewed,HPAT2_ARATH,Hydroxyproline O-arabinosyltransferase 2 (EC 2...,HPAT2 At2g25260 T22F11.15,Arabidopsis thaliana (Mouse-ear cress),3702.0,2.4.2.58,358.0,MGFRGKYFFPILMTLSLFLIIRYNYIVSDDPPLRQELPGRRSASSG...,True,True
1,rule0001_88,2,Q84IF9,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...,Q84IF9,True,unreviewed,Q84IF9_GEOSE,Cysteine synthase (EC 2.5.1.47),NaN,Geobacillus stearothermophilus (Bacillus stear...,1422.0,2.5.1.47,308.0,MARTVNSITELIGDTPAVKLNRIVDEDSADVYLKLEFMNPGSSVKD...,True,True
2,rule0001_88,3,Q9FY51,Any;Any,Any;Any,[#6;!$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8...,Q9FY51,True,reviewed,HPAT3_ARATH,Hydroxyproline O-arabinosyltransferase 3 (EC 2...,HPAT3 At5g13500 T6I14.30,Arabidopsis thaliana (Mouse-ear cress),3702.0,2.4.2.58,358.0,MGKASGLLLFLLGFGFFVVTYNLLTLIVHNRSGVSNSDGSPLLDPV...,True,True
3,rule0003_170,1,A1CFL1,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...,A1CFL1,True,reviewed,PATD_ASPCL,Alcohol dehydrogenase patD (EC 1.1.1.-) (Patul...,patD ACLA_093590,Aspergillus clavatus (strain ATCC 1007 / CBS 5...,344612.0,1.1.1.-,388.0,MGSTLPTTYKRAFFEKQDATLTLEEVQLIEPQRGEILVKVEACGVC...,True,True
4,rule0003_170,2,A1L4Y2,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...,A1L4Y2,True,reviewed,ADHL3_ARATH,Alcohol dehydrogenase-like 3 (EC 1.1.1.1),At1g32780 F6N18.16,Arabidopsis thaliana (Mouse-ear cress),3702.0,1.1.1.1,394.0,MAETQGKVITCKAAVVWGPKVPLVIQEICVDPPQKMEVRVKILYSS...,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1939,rule0467_5,4,P13650,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,P13650,True,reviewed,DHGB_ACICA,Quinoprotein glucose dehydrogenase B (EC 1.1.5...,gdhB,Acinetobacter calcoaceticus,471.0,1.1.5.2,478.0,MNKHLLAKIALLSAVQLVTLSAFADVPLTPSQFAKAKSENFDKKVI...,True,True
1940,rule0467_5,5,P15877,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,P15877,True,reviewed,DHG_ECOLI,Quinoprotein glucose dehydrogenase (EC 1.1.5.2...,gcd b0124 JW0120,Escherichia coli (strain K12),83333.0,1.1.5.2,796.0,MAINNTGSRRLLVTLTALFAALCGLYLLIGGGWLVAIGGSWYYPIA...,True,True
1941,rule0467_5,6,P27175,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,P27175,True,reviewed,DHG_GLUOX,Quinoprotein glucose dehydrogenase (EC 1.1.5.2...,gdh GOX0265,Gluconobacter oxydans (strain 621H) (Gluconoba...,290633.0,1.1.5.2,808.0,MSTTSRPGLWALITAAVFALCGAILTVGGAWVAAIGGPLYYVILGL...,True,True
1942,rule0467_5,7,Q8ZUN8,Any;Ubiquinones_CoF,Any;Ubiquinols_CoF,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,Q8ZUN8,True,unreviewed,Q8ZUN8_PYRAE,Glucose/Sorbosone dehydrogenase domain-contain...,PAE2689,Pyrobaculum aerophilum (strain ATCC 51768 / DS...,178306.0,NaN,371.0,MRRRTFLTLAVLVSLSASLGLLTALIRKGPSEEWKFKISEVASDLE...,False,True


## 7. Summarize EC numbers for each DORAnet rule to understand what `enzyme classes` are associated with each DORAnet rule

In [60]:
def uniqueSemicolonValues(series):
    valueSet = set()

    for value in series.dropna():
        for part in str(value).split(";"):
            part = part.strip()
            if part:
                valueSet.add(part)

    return ";".join(sorted(valueSet))


def firstNonEmptyValues(series, maxValues=5):
    valueList = []

    for value in series.dropna():
        value = str(value).strip()
        if value and value not in valueList:
            valueList.append(value)
        if len(valueList) >= maxValues:
            break

    return " | ".join(valueList)


ruleEcSummaryDF = (
    ruleCandidateAnnotatedDF
    .groupby("ruleName")
    .agg(
        numCandidateQueried=("uniprotAccession", "nunique"),
        numWithEc=("hasEcNumber", "sum"),
        numWithProteinSequence=("hasProteinSequence", "sum"),
        ecNumberList=("ecNumber", uniqueSemicolonValues),
        exampleProteinNames=("proteinName", firstNonEmptyValues),
        exampleOrganisms=("organism", firstNonEmptyValues),
        ruleReactants=("ruleReactants", "first"),
        ruleProducts=("ruleProducts", "first"),
        ruleSMARTS=("ruleSMARTS", "first"),
    )
    .reset_index()
)

# Add reaction-frequency information from your DORAnet network
usedRuleFrequencyDF = (
    reactionWithRuleDF
    .groupby("ruleName")
    .agg(
        numDoranetReactions=("reactionString", "count"),
        numStarterSources=("sourceFolderNum", "nunique"),
        exampleDoranetReaction=("reactionString", "first"),
        reactionType=("reactionType", "first"),
    )
    .reset_index()
)

ruleEcSummaryDF = ruleEcSummaryDF.merge(
    usedRuleFrequencyDF,
    on="ruleName",
    how="left"
)

ruleEcSummaryDF = ruleEcSummaryDF.sort_values(
    "numDoranetReactions",
    ascending=False
)

ruleEcSummaryDF.to_csv(
    os.path.join(DNADesignResultsDir, "ruleEcSummaryDF.csv"),
    index=False
)

print(f"Rules summarized: {len(ruleEcSummaryDF):,}")

ruleEcSummaryDF[
    [
        "ruleName",
        "reactionType",
        "numDoranetReactions",
        "numCandidateQueried",
        "numWithEc",
        "ecNumberList",
        "exampleProteinNames",
        "exampleDoranetReaction",
    ]
]

Rules summarized: 64


,ruleName,reactionType,numDoranetReactions,numCandidateQueried,numWithEc,ecNumberList,exampleProteinNames,exampleDoranetReaction
20,rule0011_51,Enzymatic,3460,1,1,2.1.1.315,27-O-demethylrifamycin SV methyltransferase (D...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...
19,rule0011_50,Enzymatic,2360,1,0,,Coniferyl alcohol 9-O-methyltransferase,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...
39,rule0043_12,Enzymatic,2020,17,10,2.1.1.243;2.1.1.255;2.1.1.281;2.1.1.317,Geranyl diphosphate 2-C-methyltransferase (GPP...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...
49,rule0121_1,Enzymatic,1520,50,50,2.1.1.170;2.1.1.33,tRNA (guanine-N(7)-)-methyltransferase (EC 2.1...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...
53,rule0164_2,Enzymatic,960,50,49,1.1.-.-;1.1.1.192;1.1.1.23;1.1.1.6;3.5.4.19;3....,Histidine biosynthesis trifunctional protein [...,Cc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H](O)[C@H]2OC...
...,...,...,...,...,...,...,...,...
57,rule0183_6,Enzymatic,20,35,32,3.8.1.5;3.8.1.9,deleted | Haloalkane dehalogenase (EC 3.8.1.5)...,CC(=O)O.Cl >> CC(=O)Cl.O
58,rule0242_3,Enzymatic,20,21,20,3.5.1.4;3.5.5.1;3.5.5.7,Nitrilase | Nitrilase (EC 3.5.5.1) (NitMg) | N...,CC(=O)O.N >> CC#N.O.O
61,rule0393_9,Enzymatic,20,6,4,4.1.2.10;4.1.2.46;4.1.2.47,Hydroxynitrile lyase | (S)-hydroxynitrile lyas...,C#N.CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)n(C(C)=O)...
62,rule0394_7,Enzymatic,20,4,4,4.1.2.10;4.1.2.47,(R)-mandelonitrile lyase 1 (EC 4.1.2.10) (Hydr...,N#CC(O)(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...


### Rank candidate enzymes per rule

Each rule may have many UniProt candidates. Rank them in certain order

In [61]:
candidateRankDF = ruleCandidateAnnotatedDF.copy()

candidateRankDF["isReviewed"] = (
    candidateRankDF["uniprotReviewed"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains("reviewed|swiss-prot", na=False)
)

candidateRankDF["hasEcNumber"] = (
    candidateRankDF["ecNumber"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

candidateRankDF["hasProteinSequence"] = (
    candidateRankDF["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

candidateRankDF["proteinLengthAaNumeric"] = pd.to_numeric(
    candidateRankDF["proteinLengthAa"],
    errors="coerce"
)

candidateRankDF["hasReasonableLength"] = (
    candidateRankDF["proteinLengthAaNumeric"]
    .fillna(0)
    .between(100, 1500)
)

candidateRankDF["isBacterialProtein"] = (
    candidateRankDF["organism"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains("bacter|escherichia|bacillus|pseudomonas|streptomyces|salmonella|klebsiella", na=False)
)

candidateRankDF["candidateScore"] = (
    candidateRankDF["isReviewed"].astype(int) * 4
    + candidateRankDF["hasEcNumber"].astype(int) * 3
    + candidateRankDF["hasProteinSequence"].astype(int) * 2
    + candidateRankDF["hasReasonableLength"].astype(int) * 1
    + candidateRankDF["isBacterialProtein"].astype(int) * 1
    - candidateRankDF["candidateRankInRuleFile"] / 10000.0
)

candidateRankDF = candidateRankDF.sort_values(
    ["ruleName", "candidateScore", "candidateRankInRuleFile"],
    ascending=[True, False, True]
)

topCandidatePerRuleDF = (
    candidateRankDF
    .groupby("ruleName")
    .head(10)
    .reset_index(drop=True)
)

topCandidatePerRuleDF.to_csv(
    os.path.join(DNADesignResultsDir, "topCandidatePerRuleDF.csv"),
    index=False
)

topCandidatePerRuleDF[
    [
        "ruleName",
        "uniprotAccession",
        "candidateScore",
        "uniprotReviewed",
        "ecNumber",
        "proteinName",
        "organism",
        "proteinLengthAa",
        "candidateRankInRuleFile",
    ]
]

,ruleName,uniprotAccession,candidateScore,uniprotReviewed,ecNumber,proteinName,organism,proteinLengthAa,candidateRankInRuleFile
0,rule0001_88,Q84IF9,10.9998,unreviewed,2.5.1.47,Cysteine synthase (EC 2.5.1.47),Geobacillus stearothermophilus (Bacillus stear...,308.0,2
1,rule0001_88,Q494Q2,9.9999,reviewed,2.4.2.58,Hydroxyproline O-arabinosyltransferase 2 (EC 2...,Arabidopsis thaliana (Mouse-ear cress),358.0,1
2,rule0001_88,Q9FY51,9.9997,reviewed,2.4.2.58,Hydroxyproline O-arabinosyltransferase 3 (EC 2...,Arabidopsis thaliana (Mouse-ear cress),358.0,3
3,rule0003_170,C3JZH5,10.9993,unreviewed,1.1.1.20,Putative xanthine dehydrogenase large subunit ...,Pseudomonas fluorescens (strain SBW25),799.0,7
4,rule0003_170,O07737,10.9983,reviewed,1.1.1.1,Probable zinc-binding alcohol dehydrogenase Rv...,Mycobacterium tuberculosis (strain ATCC 25618 ...,384.0,17
...,...,...,...,...,...,...,...,...,...
516,rule0467_5,D4P700,9.9998,unreviewed,1.1.5.2,PQQ-linked membrane-bound glucose dehydrogenas...,Pantoea ananas (Erwinia uredovora),796.0,2
517,rule0467_5,I7A144,9.9997,unreviewed,1.1.5.2,Glucose dehydrogenase (EC 1.1.5.2),Thermus thermophilus,352.0,3
518,rule0467_5,Q92RB3,9.9992,unreviewed,1.1.5.2,Probable glucose dehydrogenase (Pyrroloquinoli...,Rhizobium meliloti (strain 1021) (Ensifer meli...,777.0,8
519,rule0467_5,A9XK88,6.9999,unreviewed,NaN,Cellobiose dehydrogenase,Thermothelomyces myriococcoides,828.0,1


### Print Organisms: `E.coli/Pseudomonas putida` etc

In [62]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"].astype(str).str.contains("Escherichia coli", na=False),
        "organism"].dropna().astype(str).unique())

for org in queryOrganisms:
    print(org)

Escherichia coli (strain 55989 / EAEC)
Escherichia coli (strain ATCC 8739 / DSM 1576 / NBRC 3972 / NCIMB 8545 / WDCM 00012 / Crooks)
Escherichia coli (strain K12 / DH10B)
Escherichia coli (strain K12)
Escherichia coli (strain SE11)
Escherichia coli (strain SMS-3-5 / SECEC)
Escherichia coli O139:H28 (strain E24377A / ETEC)
Escherichia coli O157:H7
Escherichia coli O157:H7 (strain EC4115 / EHEC)
Escherichia coli O1:K1 / APEC
Escherichia coli O6:H1 (strain CFT073 / ATCC 700928 / UPEC)
Escherichia coli O9:H4 (strain HS)


In [63]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"].astype(str).str.contains("Pseudomonas putida", na=False),
        "organism"].dropna().astype(str).unique())

for org in queryOrganisms:
    print(org)

Pseudomonas putida (Arthrobacter siderocapsulatus)


In [64]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"].astype(str).str.contains("Bacillus", na=False),
        "organism"].dropna().astype(str).unique())

for org in queryOrganisms:
    print(org)

Alkalihalophilus pseudofirmus (strain ATCC BAA-2126 / JCM 17055 / OF4) (Bacillus pseudofirmus)
Bacillus cytotoxicus (strain DSM 22905 / CIP 110041 / 391-98 / NVH 391-98)
Bacillus licheniformis (strain ATCC 14580 / DSM 13 / JCM 2505 / CCUG 7422 / NBRC 12200 / NCIMB 9375 / NCTC 10341 / NRRL NRS-1264 / Gibson 46)
Bacillus mycoides (strain KBAB4) (Bacillus weihenstephanensis)
Bacillus subtilis
Bacillus subtilis (strain 168)
Bacillus thuringiensis (strain Al Hakam)
Geobacillus stearothermophilus (Bacillus stearothermophilus)


### Check which E. coli organisms are present among top ranked candidates

In [65]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"]
        .astype(str)
        .str.contains("Escherichia coli", na=False),
        "organism"
    ]
    .dropna()
    .astype(str)
    .unique()
)

print(f"Number of distinct E. coli organism labels: {len(queryOrganisms)}")

for org in queryOrganisms:
    print(org)

Number of distinct E. coli organism labels: 12
Escherichia coli (strain 55989 / EAEC)
Escherichia coli (strain ATCC 8739 / DSM 1576 / NBRC 3972 / NCIMB 8545 / WDCM 00012 / Crooks)
Escherichia coli (strain K12 / DH10B)
Escherichia coli (strain K12)
Escherichia coli (strain SE11)
Escherichia coli (strain SMS-3-5 / SECEC)
Escherichia coli O139:H28 (strain E24377A / ETEC)
Escherichia coli O157:H7
Escherichia coli O157:H7 (strain EC4115 / EHEC)
Escherichia coli O1:K1 / APEC
Escherichia coli O6:H1 (strain CFT073 / ATCC 700928 / UPEC)
Escherichia coli O9:H4 (strain HS)


### count how many rules have at least one E. coli candidate

In [66]:
topCandidatePerRuleDF["isEscherichiaColi"] = (
    topCandidatePerRuleDF["organism"]
    .fillna("")
    .astype(str)
    .str.contains("Escherichia coli", na=False)
)

ecoliCandidateSummaryDF = (
    topCandidatePerRuleDF
    .groupby("ruleName")
    .agg(
        numTopCandidates=("uniprotAccession", "nunique"),
        numEcoliCandidates=("isEscherichiaColi", "sum"),
    )
    .reset_index()
)

ecoliCandidateSummaryDF["hasEcoliCandidate"] = (
    ecoliCandidateSummaryDF["numEcoliCandidates"] > 0
)

print(ecoliCandidateSummaryDF["hasEcoliCandidate"].value_counts(dropna=False))

ecoliCandidateSummaryDF.head()

hasEcoliCandidate
False    39
True     25
Name: count, dtype: int64


,ruleName,numTopCandidates,numEcoliCandidates,hasEcoliCandidate
0,rule0001_88,3,0,False
1,rule0003_170,10,0,False
2,rule0003_175,10,0,False
3,rule0003_176,10,0,False
4,rule0003_177,7,1,True


### 8. Automatically select the best E. coli enzyme per rule

This selects the highest-scoring E. coli candidate for each DORAnet rule

In [67]:
candidateRankDF["isEscherichiaColi"] = (
    candidateRankDF["organism"]
    .fillna("")
    .astype(str)
    .str.contains("Escherichia coli", na=False)
)

ecoliCandidateDF = candidateRankDF[
    candidateRankDF["isEscherichiaColi"]
].copy()

ecoliCandidateDF = ecoliCandidateDF.sort_values(
    ["ruleName", "candidateScore", "candidateRankInRuleFile"],
    ascending=[True, False, True]
)

selectedEcoliEnzymeDF = (
    ecoliCandidateDF
    .groupby("ruleName")
    .head(1)
    .reset_index(drop=True)
)

selectedEcoliEnzymeDF["selectForDnaDesign"] = "yes"
selectedEcoliEnzymeDF["selectionMode"] = "automatic_ecoli_only"
selectedEcoliEnzymeDF["enzymeConfidence"] = "automatic_needs_later_review"
selectedEcoliEnzymeDF["selectionReason"] = (
    "Automatically selected highest-scoring Escherichia coli candidate per DORAnet rule"
)

selectedEcoliEnzymeDF.to_csv(
    os.path.join(DNADesignResultsDir, "selectedEcoliEnzymeDF.csv"),
    index=False
)

print(f"Rules with selected E. coli enzyme: {selectedEcoliEnzymeDF['ruleName'].nunique():,}")
print(f"Selected enzyme rows             : {len(selectedEcoliEnzymeDF):,}")

selectedEcoliEnzymeDF[
    [
        "ruleName",
        "uniprotAccession",
        "candidateScore",
        "uniprotReviewed",
        "ecNumber",
        "proteinName",
        "organism",
        "proteinLengthAa",
        "selectionMode",
    ]
].head(30)

Rules with selected E. coli enzyme: 26
Selected enzyme rows             : 26


,ruleName,uniprotAccession,candidateScore,uniprotReviewed,ecNumber,proteinName,organism,proteinLengthAa,selectionMode
0,rule0003_177,P25906,10.9993,reviewed,1.1.1.65,Pyridoxine 4-dehydrogenase (EC 1.1.1.65),Escherichia coli (strain K12),286.0,automatic_ecoli_only
1,rule0007_174,A1A7K6,10.9999,reviewed,3.1.5.1,Deoxyguanosinetriphosphate triphosphohydrolase...,Escherichia coli O1:K1 / APEC,505.0,automatic_ecoli_only
2,rule0007_198,P13001,10.9950,reviewed,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,automatic_ecoli_only
3,rule0007_200,P13001,10.9996,reviewed,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,automatic_ecoli_only
4,rule0007_202,P13001,10.9974,reviewed,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,automatic_ecoli_only
5,rule0010_64,A1AET7,10.9990,reviewed,2.1.1.77,Protein-L-isoaspartate O-methyltransferase (EC...,Escherichia coli O1:K1 / APEC,208.0,automatic_ecoli_only
6,rule0011_48,P07364,10.9979,reviewed,2.1.1.80,Chemotaxis protein methyltransferase (EC 2.1.1...,Escherichia coli (strain K12),286.0,automatic_ecoli_only
7,rule0015_21,P0A9J6,10.9985,reviewed,2.7.1.15,Ribokinase (RK) (EC 2.7.1.15),Escherichia coli (strain K12),309.0,automatic_ecoli_only
8,rule0017_44,A1A9N7,9.9989,reviewed,3.6.1.7,Acylphosphatase (EC 3.6.1.7) (Acylphosphate ph...,Escherichia coli O1:K1 / APEC,92.0,automatic_ecoli_only
9,rule0024_52,P09029,10.9992,reviewed,6.3.4.18,N5-carboxyaminoimidazole ribonucleotide syntha...,Escherichia coli (strain K12),355.0,automatic_ecoli_only


### Check which rules do not have E. coli candidates

strict E. coli-only selection may leave some DORAnet rules without enzymes

In [68]:
allUsedRuleSet = set(
    reactionWithRuleDF.loc[
        reactionWithRuleDF["hasRuleLookup"],
        "ruleName"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

selectedEcoliRuleSet = set(
    selectedEcoliEnzymeDF["ruleName"]
    .dropna()
    .astype(str)
    .str.strip()
)

rulesWithoutEcoliSelection = sorted(
    list(allUsedRuleSet.difference(selectedEcoliRuleSet))
)

rulesWithoutEcoliSelectionDF = (
    reactionWithRuleDF[
        reactionWithRuleDF["ruleName"].isin(rulesWithoutEcoliSelection)
    ]
    .groupby("ruleName")
    .agg(
        numReactions=("reactionString", "count"),
        exampleReaction=("reactionString", "first"),
        reactionType=("reactionType", "first"),
        numCandidateUniProt=("numCandidateUniProt", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

rulesWithoutEcoliSelectionDF.to_csv(
    os.path.join(DNADesignResultsDir, "rulesWithoutEcoliSelectionDF.csv"),
    index=False
)

print(f"Rules without E. coli selection: {len(rulesWithoutEcoliSelectionDF):,}")

rulesWithoutEcoliSelectionDF

Rules without E. coli selection: 42


,ruleName,numReactions,exampleReaction,reactionType,numCandidateUniProt
15,rule0011_51,3460,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,Enzymatic,1
14,rule0011_50,2360,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,Enzymatic,1
27,rule0043_12,2020,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,Enzymatic,17
41,rule0491_2,1380,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,Enzymatic,0
12,rule0010_65,520,CO[C@@H]1[C@H](OS(=O)(=O)O)[C@@H](CO)O[C@H]1n1...,Enzymatic,1
5,rule0007_161,180,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)n(C(C)=O)cnc3...,Enzymatic,1
3,rule0003_176,160,CC(=O)O.NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP...,Enzymatic,326
30,rule0062_18,140,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,Enzymatic,2
25,rule0028_51,140,CC(=O)OC(C)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)...,Enzymatic,1
21,rule0017_16,120,O=P(O)(O)O.O=c1[nH]cnc2c1[nH+]cn2[C@@H]1O[C@H]...,Enzymatic,490


### Merge selected E. coli enzymes back to DORAnet reactions

In [69]:
selectedEnzymeDF = selectedEcoliEnzymeDF.copy()

selectedEnzymeForMergeDF = selectedEnzymeDF[
    [
        "ruleName",
        "uniprotAccession",
        "ecNumber",
        "proteinName",
        "organism",
        "uniprotReviewed",
        "proteinLengthAa",
        "proteinSequence",
        "selectionReason",
        "enzymeConfidence",
        "selectionMode",
    ]
].copy()

reactionWithSelectedEnzymeDF = reactionWithRuleDF.merge(
    selectedEnzymeForMergeDF,
    on="ruleName",
    how="left"
)

reactionWithSelectedEnzymeDF["hasSelectedEnzyme"] = (
    reactionWithSelectedEnzymeDF["uniprotAccession"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

reactionWithSelectedEnzymeDF.to_csv(
    os.path.join(DNADesignResultsDir, "reactionWithSelectedEcoliEnzymeDF.csv"),
    index=False
)

print(reactionWithSelectedEnzymeDF["hasSelectedEnzyme"].value_counts(dropna=False))

reactionWithSelectedEnzymeDF = reactionWithSelectedEnzymeDF[
    reactionWithSelectedEnzymeDF["uniprotAccession"].notna()
].copy()

reactionWithSelectedEnzymeDF[
    [
        "reactionString",
        "ruleName",
        "ecNumber",
        "uniprotAccession",
        "proteinName",
        "organism",
        "enzymeConfidence",
        "selectionMode",
    ]
].head()

hasSelectedEnzyme
False    11780
True      5620
Name: count, dtype: int64


,reactionString,ruleName,ecNumber,uniprotAccession,proteinName,organism,enzymeConfidence,selectionMode
100,CC(=O)O >> C.O=C=O,rule0024_52,6.3.4.18,P09029,N5-carboxyaminoimidazole ribonucleotide syntha...,Escherichia coli (strain K12),automatic_needs_later_review,automatic_ecoli_only
101,CC(=O)O >> C.O=C=O,rule0024_52,6.3.4.18,P09029,N5-carboxyaminoimidazole ribonucleotide syntha...,Escherichia coli (strain K12),automatic_needs_later_review,automatic_ecoli_only
102,CC(=O)O >> C.O=C=O,rule0024_52,6.3.4.18,P09029,N5-carboxyaminoimidazole ribonucleotide syntha...,Escherichia coli (strain K12),automatic_needs_later_review,automatic_ecoli_only
103,CC(=O)O >> C.O=C=O,rule0024_52,6.3.4.18,P09029,N5-carboxyaminoimidazole ribonucleotide syntha...,Escherichia coli (strain K12),automatic_needs_later_review,automatic_ecoli_only
104,CC(=O)O >> C.O=C=O,rule0024_52,6.3.4.18,P09029,N5-carboxyaminoimidazole ribonucleotide syntha...,Escherichia coli (strain K12),automatic_needs_later_review,automatic_ecoli_only


## Create selected E. coli protein-sequence table

In [70]:
selectedProteinSequenceDF = (
    selectedEcoliEnzymeDF[
        [
            "ruleName",
            "uniprotAccession",
            "ecNumber",
            "proteinName",
            "organism",
            "proteinLengthAa",
            "proteinSequence",
            "selectionMode",
            "selectionReason",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

selectedProteinSequenceDF["hasProteinSequence"] = (
    selectedProteinSequenceDF["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

selectedProteinSequenceDF.to_csv(
    os.path.join(sequenceResultsDir, "selectedEcoliProteinSequenceDF.csv"),
    index=False
)

print(selectedProteinSequenceDF["hasProteinSequence"].value_counts(dropna=False))

selectedProteinSequenceDF.head()

hasProteinSequence
True    26
Name: count, dtype: int64


,ruleName,uniprotAccession,ecNumber,proteinName,organism,proteinLengthAa,proteinSequence,selectionMode,selectionReason,hasProteinSequence
0,rule0003_177,P25906,1.1.1.65,Pyridoxine 4-dehydrogenase (EC 1.1.1.65),Escherichia coli (strain K12),286.0,MSSNTFTLGTKSVNRLGYGAMQLAGPGVFGPPRDRHVAITVLREAL...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
1,rule0007_174,A1A7K6,3.1.5.1,Deoxyguanosinetriphosphate triphosphohydrolase...,Escherichia coli O1:K1 / APEC,505.0,MAQIDFRKKINWHRRYRSPQGVKTEHEILRIFESDRGRIINSPAIR...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
2,rule0007_198,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,MNNIWWQTKGQGNVHLVLLHGWGLNAEVWRCIDEELSSHFTLHLVD...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
3,rule0007_200,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,MNNIWWQTKGQGNVHLVLLHGWGLNAEVWRCIDEELSSHFTLHLVD...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
4,rule0007_202,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,MNNIWWQTKGQGNVHLVLLHGWGLNAEVWRCIDEELSSHFTLHLVD...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True


### Summarize selected E. coli enzymes by EC class

## 9. Codon-optimize selected enzymes for E. coli using `DNA Chisel`

In [72]:
targetSpecies = "e_coli"

In [74]:
minGc = 0.30
maxGc = 0.70
gcWindow = 50
stopCodon = "TAA"

forbiddenPatternList = [
    "BsaI_site",
    "BsmBI_site",
    "EcoRI_site",
    "XbaI_site",
    "SpeI_site",
    "PstI_site",
    "NotI_site",
]


def buildSimpleReverseCodonMap():
    standardTable = CodonTable.unambiguous_dna_by_name["Standard"]

    aaToCodons = {}
    for codon, aa in standardTable.forward_table.items():
        aaToCodons.setdefault(aa, []).append(codon)

    preferredCodonDict = {
        aa: sorted(codons)[0]
        for aa, codons in aaToCodons.items()
    }

    preferredCodonDict["M"] = "ATG"
    preferredCodonDict["W"] = "TGG"

    return preferredCodonDict


def reverseTranslateProtein(proteinSequence):
    preferredCodonDict = buildSimpleReverseCodonMap()

    cleanProteinSequence = (
        str(proteinSequence)
        .replace("*", "")
        .replace(" ", "")
        .replace("\n", "")
        .upper()
    )

    codonList = []

    for aa in cleanProteinSequence:
        if aa not in preferredCodonDict:
            raise ValueError(f"Cannot reverse translate amino acid: {aa}")
        codonList.append(preferredCodonDict[aa])

    return "".join(codonList)


def optimizeCdsForEcoli(initialCds):
    sequenceLength = len(initialCds)
    geneLocation = (0, sequenceLength)

    constraints = [
        EnforceTranslation(location=geneLocation),
        EnforceGCContent(mini=minGc, maxi=maxGc, window=gcWindow),
    ]

    for patternName in forbiddenPatternList:
        constraints.append(AvoidPattern(patternName))

    objectives = [
        MaximizeCAI(species=targetSpecies, location=geneLocation)
    ]

    optimizationProblem = DnaOptimizationProblem(
        sequence=initialCds,
        constraints=constraints,
        objectives=objectives,
    )

    optimizationProblem.resolve_constraints()
    optimizationProblem.optimize()

    return optimizationProblem.sequence, optimizationProblem


proteinForDesignDF = selectedProteinSequenceDF[
    selectedProteinSequenceDF["hasProteinSequence"]
].copy()

optimizedGeneRecords = []

for _, row in tqdm(
    proteinForDesignDF.iterrows(),
    total=len(proteinForDesignDF),
    desc="Optimizing E. coli selected genes"
):
    ruleName = row["ruleName"]
    uniprotAccession = row["uniprotAccession"]
    proteinSequence = row["proteinSequence"]

    initialCds = reverseTranslateProtein(proteinSequence)
    optimizedCds, optimizationProblem = optimizeCdsForEcoli(initialCds)
    finalCds = optimizedCds + stopCodon

    optimizedGeneRecords.append({
        "ruleName": ruleName,
        "uniprotAccession": uniprotAccession,
        "ecNumber": row["ecNumber"],
        "proteinName": row["proteinName"],
        "sourceOrganism": row["organism"],
        "expressionHost": "Escherichia coli",
        "targetSpecies": targetSpecies,
        "proteinLengthAa": len(str(proteinSequence)),
        "initialCdsLengthBp": len(initialCds),
        "optimizedCdsLengthBp": len(finalCds),
        "optimizedCds": finalCds,
        "selectionMode": row["selectionMode"],
        "selectionReason": row["selectionReason"],
        "constraintsSummary": optimizationProblem.constraints_text_summary(),
        "objectivesSummary": optimizationProblem.objectives_text_summary(),
    })

optimizedEcoliGeneDF = pd.DataFrame(optimizedGeneRecords)

optimizedEcoliGeneDF.to_csv(
    os.path.join(dnaResultsDir, "optimizedEcoliGeneDF.csv"),
    index=False
)

with open(os.path.join(dnaResultsDir, "optimizedEcoliGenes.fasta"), "w", encoding="utf-8") as f:
    for _, row in optimizedEcoliGeneDF.iterrows():
        fastaHeader = (
            f">{row['ruleName']}|{row['uniprotAccession']}|"
            f"EC={row['ecNumber']}|host=Escherichia_coli"
        )
        f.write(fastaHeader + "\n")
        f.write(str(row["optimizedCds"]) + "\n\n")

print(f"Optimized genes: {len(optimizedEcoliGeneDF):,}")

optimizedEcoliGeneDF.head()

constraint:   0%|                                                                  | 0/8 [00:00<?, ?it/s, now=AvoidPattern[0-858](patte...]

location:   0%|                                                                                            | 0/1 [00:00<?, ?it/s, now=None]

location:   0%|                                                                                      | 0/1 [00:00<?, ?it/s, now=388-394(-)]

                                                                                                                                           

location:   0%|                                                                                      | 0/1 [00:00<?, ?it/s, now=388-394(-)]

location:   0%|                                                                                      | 0/1 [00:00<?, ?it/s, now=461-467(+)]

                                                                                                                                           

location:   0

Optimized genes: 26


,ruleName,uniprotAccession,ecNumber,proteinName,sourceOrganism,expressionHost,targetSpecies,proteinLengthAa,initialCdsLengthBp,optimizedCdsLengthBp,optimizedCds,selectionMode,selectionReason,constraintsSummary,objectivesSummary
0,rule0003_177,P25906,1.1.1.65,Pyridoxine 4-dehydrogenase (EC 1.1.1.65),Escherichia coli (strain K12),Escherichia coli,e_coli,286,858,861,ATGAGCAGCAACACCTTTACCCTGGGCACCAAAAGCGTGAACCGCC...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -10.95\n -1...
1,rule0007_174,A1A7K6,3.1.5.1,Deoxyguanosinetriphosphate triphosphohydrolase...,Escherichia coli O1:K1 / APEC,Escherichia coli,e_coli,505,1515,1518,ATGGCGCAGATTGATTTTCGCAAAAAAATTAACTGGCATCGCCGCT...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -9.74\n -...
2,rule0007_198,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),Escherichia coli,e_coli,256,768,771,ATGAACAACATTTGGTGGCAGACCAAAGGCCAGGGCAACGTGCATC...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -7.53\n -...
3,rule0007_200,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),Escherichia coli,e_coli,256,768,771,ATGAACAACATTTGGTGGCAGACCAAAGGCCAGGGCAACGTGCATC...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -7.53\n -...
4,rule0007_202,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),Escherichia coli,e_coli,256,768,771,ATGAACAACATTTGGTGGCAGACCAAAGGCCAGGGCAACGTGCATC...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -7.53\n -...


### Merge optimized E. coli DNA back to reactions

In [75]:
optimizedGeneForMergeDF = optimizedEcoliGeneDF[
    [
        "ruleName",
        "uniprotAccession",
        "optimizedCds",
        "optimizedCdsLengthBp",
        "expressionHost",
        "targetSpecies",
    ]
].copy()

reactionWithEcoliDnaDesignDF = reactionWithSelectedEnzymeDF.merge(
    optimizedGeneForMergeDF,
    on=["ruleName", "uniprotAccession"],
    how="left"
)

reactionWithEcoliDnaDesignDF["hasOptimizedDna"] = (
    reactionWithEcoliDnaDesignDF["optimizedCds"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

reactionWithEcoliDnaDesignDF.to_csv(
    os.path.join(dnaResultsDir, "reactionWithEcoliDnaDesignDF.csv"),
    index=False
)

print(reactionWithEcoliDnaDesignDF["hasOptimizedDna"].value_counts(dropna=False))
reactionWithEcoliDnaDesignDF

hasOptimizedDna
True    5620
Name: count, dtype: int64


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,proteinSequence,selectionReason,enzymeConfidence,selectionMode,hasSelectedEnzyme,optimizedCds,optimizedCdsLengthBp,expressionHost,targetSpecies,hasOptimizedDna
0,CC(=O)O,C.O=C=O,CC(=O)O >> C.O=C=O,rule0024_52,No_Thermo,"(1,)","(1, 1)",Enzymatic,1,2,...,MKQVCVLGNGQLGRMLRQAGEPLGIAVWPVGLDAEPAAVPFQQSVI...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAAACAGGTGTGCGTGCTGGGCAACGGCCAGCTGGGCCGCATGC...,1068,Escherichia coli,e_coli,True
1,CC(=O)O,C.O=C=O,CC(=O)O >> C.O=C=O,rule0024_52,No_Thermo,"(1,)","(1, 1)",Enzymatic,1,2,...,MKQVCVLGNGQLGRMLRQAGEPLGIAVWPVGLDAEPAAVPFQQSVI...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAAACAGGTGTGCGTGCTGGGCAACGGCCAGCTGGGCCGCATGC...,1068,Escherichia coli,e_coli,True
2,CC(=O)O,C.O=C=O,CC(=O)O >> C.O=C=O,rule0024_52,No_Thermo,"(1,)","(1, 1)",Enzymatic,1,2,...,MKQVCVLGNGQLGRMLRQAGEPLGIAVWPVGLDAEPAAVPFQQSVI...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAAACAGGTGTGCGTGCTGGGCAACGGCCAGCTGGGCCGCATGC...,1068,Escherichia coli,e_coli,True
3,CC(=O)O,C.O=C=O,CC(=O)O >> C.O=C=O,rule0024_52,No_Thermo,"(1,)","(1, 1)",Enzymatic,1,2,...,MKQVCVLGNGQLGRMLRQAGEPLGIAVWPVGLDAEPAAVPFQQSVI...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAAACAGGTGTGCGTGCTGGGCAACGGCCAGCTGGGCCGCATGC...,1068,Escherichia coli,e_coli,True
4,CC(=O)O,C.O=C=O,CC(=O)O >> C.O=C=O,rule0024_52,No_Thermo,"(1,)","(1, 1)",Enzymatic,1,2,...,MKQVCVLGNGQLGRMLRQAGEPLGIAVWPVGLDAEPAAVPFQQSVI...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAAACAGGTGTGCGTGCTGGGCAACGGCCAGCTGGGCCGCATGC...,1068,Escherichia coli,e_coli,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5615,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@]1(C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,rule0126_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,MSTPLQGIKVLDFTGVQSGPSCTQMLAWFGADVIKIERPGVGDVTR...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAGCACCCCGCTGCAAGGCATTAAAGTGCTGGATTTTACCGGCG...,1251,Escherichia coli,e_coli,True
5616,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@]1(C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,rule0126_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,MSTPLQGIKVLDFTGVQSGPSCTQMLAWFGADVIKIERPGVGDVTR...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAGCACCCCGCTGCAAGGCATTAAAGTGCTGGATTTTACCGGCG...,1251,Escherichia coli,e_coli,True
5617,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@]1(C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,rule0126_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,MSTPLQGIKVLDFTGVQSGPSCTQMLAWFGADVIKIERPGVGDVTR...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAGCACCCCGCTGCAAGGCATTAAAGTGCTGGATTTTACCGGCG...,1251,Escherichia coli,e_coli,True
5618,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@]1(C...,CC(O)O[C@H]1[C@H]2Oc3nc4c(ncn4[C@@H]2O[C@@H]1C...,rule0126_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,MSTPLQGIKVLDFTGVQSGPSCTQMLAWFGADVIKIERPGVGDVTR...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAGCACCCCGCTGCAAGGCATTAAAGTGCTGGATTTTACCGGCG...,1251,Escherichia coli,e_coli,True


### 10. Export E. coli design files for `Teselagen`

In [76]:
ecoliPartImportDF = optimizedEcoliGeneDF.copy()

ecoliPartImportDF["partName"] = (
    ecoliPartImportDF["ruleName"].astype(str)
    + "__"
    + ecoliPartImportDF["uniprotAccession"].astype(str)
    + "__EcoliHost"
)

ecoliPartImportDF["partType"] = "CDS"
ecoliPartImportDF["sequenceType"] = "DNA"
ecoliPartImportDF["assemblyMethod"] = "GoldenGate_or_Gibson"

ecoliPartImportDF["description"] = (
    "DORAnet rule: "
    + ecoliPartImportDF["ruleName"].astype(str)
    + "; EC: "
    + ecoliPartImportDF["ecNumber"].astype(str)
    + "; protein: "
    + ecoliPartImportDF["proteinName"].astype(str)
    + "; source organism: "
    + ecoliPartImportDF["sourceOrganism"].astype(str)
    + "; expression host: Escherichia coli"
)

ecoliPartImportCols = [
    "partName",
    "partType",
    "sequenceType",
    "optimizedCds",
    "description",
    "assemblyMethod",
    "ruleName",
    "uniprotAccession",
    "ecNumber",
    "proteinName",
    "sourceOrganism",
    "expressionHost",
    "targetSpecies",
    "optimizedCdsLengthBp",
]

ecoliPartImportDF[ecoliPartImportCols].to_csv(
    os.path.join(handoffResultsDir, "teselagenEcoliPartImportDF.csv"),
    index=False
)

with open(os.path.join(handoffResultsDir, "ecoliPathwayParts.fasta"), "w", encoding="utf-8") as f:
    for _, row in ecoliPartImportDF.iterrows():
        f.write(f">{row['partName']}\n")
        f.write(str(row["optimizedCds"]) + "\n\n")

print("Saved:")
print(os.path.join(handoffResultsDir, "teselagenEcoliPartImportDF.csv"))
print(os.path.join(handoffResultsDir, "ecoliPathwayParts.fasta"))

ecoliPartImportDF[ecoliPartImportCols].head()

Saved:
/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/07_webtool_handoff/teselagenEcoliPartImportDF.csv
/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/07_webtool_handoff/ecoliPathwayParts.fasta


,partName,partType,sequenceType,optimizedCds,description,assemblyMethod,ruleName,uniprotAccession,ecNumber,proteinName,sourceOrganism,expressionHost,targetSpecies,optimizedCdsLengthBp
0,rule0003_177__P25906__EcoliHost,CDS,DNA,ATGAGCAGCAACACCTTTACCCTGGGCACCAAAAGCGTGAACCGCC...,DORAnet rule: rule0003_177; EC: 1.1.1.65; prot...,GoldenGate_or_Gibson,rule0003_177,P25906,1.1.1.65,Pyridoxine 4-dehydrogenase (EC 1.1.1.65),Escherichia coli (strain K12),Escherichia coli,e_coli,861
1,rule0007_174__A1A7K6__EcoliHost,CDS,DNA,ATGGCGCAGATTGATTTTCGCAAAAAAATTAACTGGCATCGCCGCT...,DORAnet rule: rule0007_174; EC: 3.1.5.1; prote...,GoldenGate_or_Gibson,rule0007_174,A1A7K6,3.1.5.1,Deoxyguanosinetriphosphate triphosphohydrolase...,Escherichia coli O1:K1 / APEC,Escherichia coli,e_coli,1518
2,rule0007_198__P13001__EcoliHost,CDS,DNA,ATGAACAACATTTGGTGGCAGACCAAAGGCCAGGGCAACGTGCATC...,DORAnet rule: rule0007_198; EC: 3.1.1.85; prot...,GoldenGate_or_Gibson,rule0007_198,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),Escherichia coli,e_coli,771
3,rule0007_200__P13001__EcoliHost,CDS,DNA,ATGAACAACATTTGGTGGCAGACCAAAGGCCAGGGCAACGTGCATC...,DORAnet rule: rule0007_200; EC: 3.1.1.85; prot...,GoldenGate_or_Gibson,rule0007_200,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),Escherichia coli,e_coli,771
4,rule0007_202__P13001__EcoliHost,CDS,DNA,ATGAACAACATTTGGTGGCAGACCAAAGGCCAGGGCAACGTGCATC...,DORAnet rule: rule0007_202; EC: 3.1.1.85; prot...,GoldenGate_or_Gibson,rule0007_202,P13001,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),Escherichia coli,e_coli,771


In [77]:
ecoliPipelineSummary = {
    "totalReactionRows": len(reactionWithRuleDF),
    "rulesWithRuleLookup": reactionWithRuleDF["hasRuleLookup"].sum(),
    "uniqueRulesWithRuleLookup": reactionWithRuleDF.loc[
        reactionWithRuleDF["hasRuleLookup"], "ruleName"
    ].nunique(),
    "rulesWithSelectedEcoliEnzyme": selectedEcoliEnzymeDF["ruleName"].nunique(),
    "selectedEcoliEnzymes": len(selectedEcoliEnzymeDF),
    "selectedProteinsWithSequence": selectedProteinSequenceDF["hasProteinSequence"].sum(),
    "optimizedEcoliGenes": len(optimizedEcoliGeneDF),
    "reactionRowsWithOptimizedDna": reactionWithEcoliDnaDesignDF["hasOptimizedDna"].sum(),
}

ecoliPipelineSummaryDF = pd.DataFrame(
    list(ecoliPipelineSummary.items()),
    columns=["metric", "value"]
)


ecoliPipelineSummaryDF

,metric,value
0,totalReactionRows,17400
1,rulesWithRuleLookup,17400
2,uniqueRulesWithRuleLookup,68
3,rulesWithSelectedEcoliEnzyme,26
4,selectedEcoliEnzymes,26
5,selectedProteinsWithSequence,26
6,optimizedEcoliGenes,26
7,reactionRowsWithOptimizedDna,5620
